# Module 2: Feature Engineering & Preventing Data Leakage
### Course: Applied Data Science & Machine Learning in Production
**Case Study:** Customer Churn & Lifetime Value (LTV) Prediction for a Subscription Business

---
### Learning Objectives:
1. **Detect & Prevent Data Leakage:** Master the taxonomy of leakage (lookahead bias, preprocessor leakage, target proxy contamination).
2. **Temporal Splitting:** Partition time-series subscription telemetry chronologically rather than with random shuffle.
3. **Custom Scikit-Learn Transformers:** Author production-ready, reusable transformers with strict `fit` and `transform` contracts.
4. **Leakage Auditing:** Build automated assertion suites that flag suspicious feature-target correlations and data contamination.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

DATA_PATH = Path("../data/raw/subscriptions_raw.csv")
df = pd.read_csv(DATA_PATH, parse_dates=["signup_date"])
print(f"Loaded {df.shape[0]} records from {DATA_PATH.name}")

## 1. Temporal Partitioning vs Random Shuffling
In subscription businesses, observations have a temporal arrow. Using standard random $k$-fold cross-validation or random train-test splitting introduces **lookahead bias**, where future trends leak into historical training data.

In [ ]:
# SOLUTION: Temporal Split Implementation
def temporal_train_test_split(data, date_col="signup_date", split_ratio=0.80):
    sorted_df = data.sort_values(by=date_col).reset_index(drop=True)
    split_index = int(len(sorted_df) * split_ratio)
    
    train_data = sorted_df.iloc[:split_index].copy()
    test_data = sorted_df.iloc[split_index:].copy()
    return train_data, test_data

train_df, test_df = temporal_train_test_split(df, split_ratio=0.80)
print(f"Train set: {len(train_df)} rows | Date range: {train_df['signup_date'].min().date()} -> {train_df['signup_date'].max().date()}")
print(f"Test set:  {len(test_df)} rows | Date range: {test_df['signup_date'].min().date()} -> {test_df['signup_date'].max().date()}")

# Strict temporal assertion
assert train_df['signup_date'].max() <= test_df['signup_date'].min(), "Temporal Leakage detected!"
print("Temporal split verified: Zero lookahead overlap!")

## 2. Production Feature Engineering: Custom Transformers
We formulate domain-driven behavioral interaction ratios:
- **Charge-to-Tenure Ratio:** $\frac{\text{monthly\_charges}}{\text{tenure} + 1}$ (measures immediate cost intensity).
- **Ticket Velocity:** $\frac{\text{num\_support\_tickets}}{\text{tenure} + 1}$ (frustration frequency).
- **Engagement Health Index:** $\text{avg\_weekly\_logins} \times \max(0, 1 - \text{usage\_drop\_pct\_last\_30d})$.

In [ ]:
# SOLUTION: Domain Behavioral Feature Transformer
class DomainBehavioralTransformer(BaseEstimator, TransformerMixin):
    """Computes subscription interaction ratios without lookahead leakage."""
    
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        X_out = X.copy()
        tenure = X_out["tenure_months"].values
        monthly = X_out["monthly_charges"].values
        tickets = X_out["num_support_tickets"].values
        logins = X_out["avg_weekly_logins"].values
        drop_pct = X_out["usage_drop_pct_last_30d"].values
        
        X_out["charge_to_tenure_ratio"] = monthly / (tenure + 1.0)
        X_out["ticket_velocity"] = tickets / (tenure + 1.0)
        X_out["engagement_health_index"] = logins * np.maximum(0.0, 1.0 - drop_pct)
        return X_out

feat_eng = DomainBehavioralTransformer()
engineered_sample = feat_eng.transform(train_df.head(5))
display(engineered_sample[["customer_id", "charge_to_tenure_ratio", "ticket_velocity", "engagement_health_index"]])

## 3. End-to-End Leak-Free Preprocessing Pipeline
We integrate continuous scaling, one-hot categorical encoding, and custom feature synthesis into a single Scikit-Learn `Pipeline`. Preprocessor parameters are learned **exclusively** on the training split.

In [ ]:
numerical_features = [
    "tenure_months", "monthly_charges", "total_charges",
    "num_support_tickets", "avg_weekly_logins", "usage_drop_pct_last_30d",
    "charge_to_tenure_ratio", "ticket_velocity", "engagement_health_index"
]
categorical_features = ["contract_type", "payment_method", "internet_service"]
binary_features = ["has_premium_support", "multi_seat_license"]

col_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("bin", "passthrough", binary_features)
    ]
)

preprocessor = Pipeline([
    ("domain_features", DomainBehavioralTransformer()),
    ("col_transform", col_transformer)
])

# Prepare feature matrices (strictly excluding identifiers and targets)
features_to_drop = ["customer_id", "signup_date", "churn", "customer_ltv"]
X_train = train_df.drop(columns=features_to_drop)
y_train_churn = train_df["churn"].values

X_test = test_df.drop(columns=features_to_drop)
y_test_churn = test_df["churn"].values

# Fit ONLY on train; transform both
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print(f"X_train preprocessed shape: {X_train_proc.shape}")
print(f"X_test preprocessed shape:  {X_test_proc.shape}")

## 4. Leakage Auditing Suite
Before touching any model training algorithm, we run an automated leakage diagnostic.

In [ ]:
def audit_leakage(train_data, target_col="churn", threshold=0.85):
    flags = []
    numeric_cols = train_data.select_dtypes(include=[np.number]).columns
    for c in numeric_cols:
        if c != target_col:
            corr = train_data[c].corr(train_data[target_col])
            if abs(corr) >= threshold:
                flags.append(f"HIGH CORRELATION LEAK: '{c}' has |r| = {abs(corr):.3f}")
    if not flags:
        print("✓ Leakage Audit Passed: No suspicious feature-target correlations detected.")
    else:
        for f in flags:
            print("⚠ ALERT:", f)

audit_leakage(train_df)

## 5. Summary & Next Steps
- Preprocessing steps must never be fit across the entire dataset.
- Modular pipelines guarantee zero data leakage between development and serving time.
- Next module: **Imbalanced Learning, Model Calibration & Cost-Benefit Threshold Optimization**.